# Phase 5p: Working CLAP checkpoint (`laion/larger_clap_general`) under the primary protocol

Phase 2d showed that the HF `laion/larger_clap_music` conversion used for every earlier CLAP result is defective
(zero-shot genre and mood at chance, unrelated text prompts at cosine 0.999; LAION-AI/CLAP issue #126).
`outputs/clap_checkpoint_validation.csv` picked `laion/larger_clap_general` as the best working HF checkpoint
(zero-shot genre 0.57 vs chance 0.14), and `extract_clap_general.py` embedded all 3,232 excerpts with it
(fixed 10-s windows, L2-mean, as in Phase 2b).

This notebook asks whether the CLAP conclusions change with a working checkpoint, under the artist-filtered primary
protocol (Phase 5j):
1. configuration (raw / z; MuQ layer) chosen on filtered validation by quadrant P@10;
2. test retrieval (P@10, nDCG@10) with bootstrap CIs, paired Wilcoxon (Pratt) with Holm within split × metric,
   and TOST against SBERT and MuQ (margins ±0.05 P@10, ±0.03 nDCG@10);
3. late fusion with SBERT (α on validation) vs lyrics alone and vs MuQ + SBERT;
4. ridge AV and logistic quadrant probes (70/15/15);
5. same-artist share of unfiltered top-10s.

All outputs are new files with a `clap_general_` prefix; nothing earlier is overwritten.

In [1]:
import os
import re
import unicodedata
import warnings
import numpy as np
import pandas as pd
from scipy.stats import wilcoxon, t as t_dist
from sklearn.linear_model import RidgeCV, LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import r2_score, f1_score

warnings.filterwarnings('ignore')

ROOT = os.environ.get('MUSIC_RESEARCH_ROOT', os.path.abspath('.'))
OUT_DIR = os.path.join(ROOT, 'outputs')
ENC_DIR = os.path.join(OUT_DIR, 'encoders')
SIGMA, K, N_BOOT, SEED = 0.10, 10, 10_000, 42
ALPHAS = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
SCHEMES = {'tvt_70_15_15': '70/15/15', 'tvt_40_30_30': '40/30/30'}
MARGIN = {'quad_precision': 0.05, 'ndcg': 0.03}
rng = np.random.default_rng(SEED)

lyrics_df = pd.read_parquet(os.path.join(OUT_DIR, 'lyrics_features.parquet'))
merged    = pd.read_parquet(os.path.join(OUT_DIR, 'merged_dataset.parquet'))
cat_sids  = lyrics_df['song_id'].tolist()
cat_pos   = {s: i for i, s in enumerate(cat_sids)}

a_ids = pd.read_csv(os.path.join(ENC_DIR, 'audio_song_ids.csv'))['song_id'].tolist()
a_row = [{s: i for i, s in enumerate(a_ids)}[s] for s in cat_sids]
a_old = pd.read_parquet(os.path.join(OUT_DIR, 'audio_features.parquet')).set_index('song_id').loc[cat_sids]
a_det = pd.read_parquet(os.path.join(OUT_DIR, 'audio_features_clap_det.parquet')).set_index('song_id').loc[cat_sids]
a_gen = pd.read_parquet(os.path.join(OUT_DIR, 'audio_features_clap_general.parquet')).set_index('song_id').loc[cat_sids]
print('checkpoint:', a_gen.checkpoint.iloc[0], '@', a_gen.checkpoint_commit.iloc[0],
      '| windows per track:', a_gen.n_windows.value_counts().to_dict())
muq = np.load(os.path.join(ENC_DIR, 'audio_muq.npy'))[a_row].astype(np.float64)

REPS = {'clap_general': np.vstack(a_gen['audio_clap_gen'].values).astype(np.float64),
        'clap_music_rand': np.vstack(a_old['audio_clap'].values).astype(np.float64),
        'clap_music_det': np.vstack(a_det['audio_clap_det'].values).astype(np.float64),
        'handcrafted': np.vstack(a_old['audio_handcrafted'].values).astype(np.float64)}
for L in range(13):
    REPS[f'muq_L{L}'] = muq[:, L]
assert pd.read_csv(os.path.join(ENC_DIR, 'text_song_ids.csv'))['song_id'].tolist() == cat_sids
REPS['sbert'] = np.load(os.path.join(ENC_DIR, 'text_sbert_mpnet_trunc.npy')).astype(np.float64)
FAMILY = lambda r: r.split('_L')[0]
FAMS = ['clap_general', 'clap_music_rand', 'clap_music_det', 'handcrafted', 'muq', 'sbert']

lab  = merged.set_index('song_id').loc[cat_sids]
quad = lab['quadrant'].values
Y    = lab[['arousal', 'valence']].values.astype(np.float64)
D_AV = np.linalg.norm(Y[:, None] - Y[None], axis=-1)

mm = merged[merged['is_multimodal']]
lyr_stem_to_sid = {os.path.splitext(os.path.basename(p))[0]: s for s, p in zip(mm['song_id'], mm['lyrics_path'])}
def read_split(modality, scheme, subset):
    folder = 'MERGE_Audio_Balanced' if modality == 'audio' else 'MERGE_Lyrics_Balanced'
    ids = pd.read_csv(os.path.join(ROOT, folder, 'tvt_dataframes', scheme,
                                   f'{scheme}_{subset}_{modality}_balanced.csv'))['Song']
    return set(ids.map(lyr_stem_to_sid).dropna()) if modality == 'lyrics' else set(ids)
cat = set(cat_sids)
shared = {lbl: {sub: sorted(read_split('audio', sc, sub) & read_split('lyrics', sc, sub) & cat)
                for sub in ['train', 'validate', 'test']} for sc, lbl in SCHEMES.items()}
print({lbl: {k: len(v) for k, v in d.items()} for lbl, d in shared.items()})

def _norm_artist(s):
    s = unicodedata.normalize('NFKD', str(s)).encode('ascii', 'ignore').decode().lower().strip()
    return re.sub(r'[^a-z0-9]+', ' ', re.sub(r'^the\s+', '', s)).strip()
ART_ID = np.array([_norm_artist(a) for a in lab['artist'].values])
ART = ART_ID[:, None] == ART_ID[None]
np.fill_diagonal(ART, False)
print(f'{len(set(ART_ID))} artist groups')

checkpoint: laion/larger_clap_general @ ada0c23a36c4e8582805bb38fec3905903f18b41 | windows per track: {3: 2074, 1: 1}


{'70/15/15': {'train': 1187, 'validate': 305, 'test': 302}, '40/30/30': {'train': 475, 'validate': 616, 'test': 617}}


1086 artist groups


## 1. Retrieval machinery and validation selection

In [2]:
def prep(X, how):
    X = StandardScaler().fit_transform(X) if how == 'z' else X
    return X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-12)

_cache = {}
def unit(rep, how):
    if (rep, how) not in _cache:
        _cache[(rep, how)] = prep(REPS[rep], how)
    return _cache[(rep, how)]

DISC = 1 / np.log2(np.arange(2, K + 2))
def metrics_from_scores(S, rows, artist_filter=True):
    S = S.copy(); S[np.arange(len(rows)), rows] = -np.inf
    Dq = D_AV[rows].copy(); Dq[np.arange(len(rows)), rows] = np.inf
    if artist_filter:
        S[ART[rows]] = -np.inf; Dq[ART[rows]] = np.inf
    top = np.argpartition(-S, K, 1)[:, :K]
    top = np.take_along_axis(top, np.take_along_axis(S, top, 1).argsort(1)[:, ::-1], 1)
    d = np.take_along_axis(D_AV[rows], top, 1)
    idcg = (np.exp(-np.sort(Dq, 1)[:, :K]**2 / (2 * SIGMA**2)) * DISC).sum(1)
    return {'quad_precision': (quad[top] == quad[rows][:, None]).mean(1),
            'ndcg': (np.exp(-d**2 / (2 * SIGMA**2)) * DISC).sum(1) / idcg,
            'same_artist': ART[rows[:, None], top].mean(1)}

def scores(cfg, rows):
    if cfg[0] == 'late':
        _, (ra, ha), (rt, ht), al = cfg
        A, T = unit(ra, ha), unit(rt, ht)
        return al * (A[rows] @ A.T) + (1 - al) * (T[rows] @ T.T)
    X = unit(*cfg)
    return X[rows] @ X.T

def run(cfg, qs, artist_filter=True):
    rows = np.array([cat_pos[s] for s in qs])
    return metrics_from_scores(scores(cfg, rows), rows, artist_filter)

val = pd.DataFrame([{'split': lbl, 'rep': rep, 'family': FAMILY(rep), 'how': how,
                     'val_quad_p10': (m := run((rep, how), shared[lbl]['validate']))['quad_precision'].mean(),
                     'val_ndcg10': m['ndcg'].mean()}
                    for lbl in SCHEMES.values() for rep in REPS for how in ['raw', 'z']])
val.to_csv(os.path.join(OUT_DIR, 'clap_general_validation_sweep.csv'), index=False)
selected = {(lbl, fam): tuple(g.sort_values('val_quad_p10', ascending=False).iloc[0][['rep', 'how']])
            for (lbl, fam), g in val.groupby(['split', 'family'])}
print(pd.DataFrame([{'split': l, 'family': f, 'config': f'{r} [{h}]'} for (l, f), (r, h) in selected.items()])
      .pivot(index='family', columns='split', values='config').loc[FAMS].to_string())

split                       40/30/30              70/15/15
family                                                    
clap_general        clap_general [z]      clap_general [z]
clap_music_rand  clap_music_rand [z]   clap_music_rand [z]
clap_music_det    clap_music_det [z]  clap_music_det [raw]
handcrafted          handcrafted [z]       handcrafted [z]
muq                       muq_L6 [z]            muq_L6 [z]
sbert                      sbert [z]             sbert [z]


## 2. Test retrieval (artist-filtered; unfiltered alongside)

In [3]:
def boot_ci(x, q=(2.5, 97.5)):
    x = np.asarray(x, float); m = x[rng.integers(0, len(x), (N_BOOT, len(x)))].mean(1)
    return np.percentile(m, q)

pq, rows = {}, []
for lbl in SCHEMES.values():
    for fam in FAMS:
        cfg = selected[(lbl, fam)]
        m = run(cfg, shared[lbl]['test']); u = run(cfg, shared[lbl]['test'], artist_filter=False)
        pq[(lbl, fam)] = m
        row = {'split': lbl, 'family': fam, 'config': f'{cfg[0]} [{cfg[1]}]', 'n': len(shared[lbl]['test'])}
        for k_ in ['quad_precision', 'ndcg']:
            lo, hi = boot_ci(m[k_]); row.update({k_: m[k_].mean(), f'{k_}_lo': lo, f'{k_}_hi': hi,
                                                 f'{k_}_unfiltered': u[k_].mean()})
        row['same_artist_share_unfiltered'] = u['same_artist'].mean()
        rows.append(row)
test_single = pd.DataFrame(rows)
test_single.to_csv(os.path.join(OUT_DIR, 'clap_general_test_retrieval.csv'), index=False)
f = lambda r, m: f"{r[m]:.3f} [{r[m+'_lo']:.3f}, {r[m+'_hi']:.3f}] ({r[m+'_unfiltered']:.3f})"
for lbl in SCHEMES.values():
    s = test_single[test_single.split == lbl]
    print(f'\n=== {lbl} shared test (n={s.n.iloc[0]}); filtered [95% CI] (unfiltered) ===')
    print(pd.DataFrame({'config': s.config.values,
                        'P@10': s.apply(lambda r: f(r, 'quad_precision'), axis=1).values,
                        'nDCG@10': s.apply(lambda r: f(r, 'ndcg'), axis=1).values,
                        'same-artist top-10 (unf.)': s.same_artist_share_unfiltered.round(3).values},
                       index=s.family.values).to_string())


=== 70/15/15 shared test (n=302); filtered [95% CI] (unfiltered) ===
                               config                          P@10                       nDCG@10  same-artist top-10 (unf.)
clap_general         clap_general [z]  0.644 [0.611, 0.677] (0.656)  0.440 [0.412, 0.469] (0.465)                      0.078
clap_music_rand   clap_music_rand [z]  0.380 [0.352, 0.408] (0.383)  0.262 [0.240, 0.285] (0.266)                      0.013
clap_music_det   clap_music_det [raw]  0.369 [0.342, 0.397] (0.372)  0.263 [0.241, 0.286] (0.267)                      0.018
handcrafted           handcrafted [z]  0.500 [0.465, 0.534] (0.506)  0.342 [0.313, 0.370] (0.350)                      0.048
muq                        muq_L6 [z]  0.610 [0.578, 0.640] (0.625)  0.417 [0.389, 0.445] (0.440)                      0.077
sbert                       sbert [z]  0.602 [0.566, 0.637] (0.605)  0.420 [0.391, 0.448] (0.427)                      0.032

=== 40/30/30 shared test (n=617); filtered [95% CI] (u

## 3. Paired comparisons and equivalence
Wilcoxon (Pratt) per query; Holm over the comparisons within split × metric. TOST: paired-t p and the bootstrap 90% CI
inside the margin.

In [4]:
def holm(p):
    p = np.asarray(p); order = np.argsort(p); m = len(p); adj = np.empty(m); run_ = 0.0
    for rank, i in enumerate(order):
        run_ = max(run_, (m - rank) * p[i]); adj[i] = min(1.0, run_)
    return adj

PAIRS = [('clap_general', 'clap_music_rand'), ('clap_general', 'clap_music_det'), ('clap_general', 'handcrafted'),
         ('clap_general', 'muq'), ('clap_general', 'sbert')]
cmp_rows = []
for lbl in SCHEMES.values():
    for metric in ['quad_precision', 'ndcg']:
        block = []
        for a, b in PAIRS:
            x, y = pq[(lbl, a)][metric], pq[(lbl, b)][metric]; d = x - y; n = len(d)
            lo, hi = boot_ci(d); lo90, hi90 = boot_ci(d, (5, 95))
            p = wilcoxon(x, y, zero_method='pratt').pvalue if (d != 0).any() else 1.0
            se = d.std(ddof=1) / np.sqrt(n); mg = MARGIN[metric]
            p_tost = max(t_dist.sf((d.mean() + mg) / se, n - 1), t_dist.cdf((d.mean() - mg) / se, n - 1))
            block.append({'split': lbl, 'metric': metric, 'a': a, 'b': b, 'diff (a − b)': d.mean(), 'ci_lo': lo,
                          'ci_hi': hi, 'p': p, 'ci90_lo': lo90, 'ci90_hi': hi90, 'margin': mg, 'p_TOST': p_tost,
                          'equivalent': (lo90 > -mg) and (hi90 < mg) and p_tost < 0.05})
        for r, ph in zip(block, holm([r['p'] for r in block])):
            r['p_holm'] = ph
        cmp_rows += block
comparisons = pd.DataFrame(cmp_rows)
comparisons.to_csv(os.path.join(OUT_DIR, 'clap_general_comparisons.csv'), index=False)
print(comparisons[['split', 'metric', 'a', 'b', 'diff (a − b)', 'ci_lo', 'ci_hi', 'p_holm', 'p_TOST', 'equivalent']]
      .to_string(index=False, float_format=lambda x: f'{x:.3g}'))

   split         metric            a               b  diff (a − b)    ci_lo  ci_hi   p_holm  p_TOST  equivalent
70/15/15 quad_precision clap_general clap_music_rand         0.264    0.224  0.302 6.12e-26       1       False
70/15/15 quad_precision clap_general  clap_music_det         0.275    0.238  0.311 5.71e-30       1       False
70/15/15 quad_precision clap_general     handcrafted         0.144    0.111  0.177 1.04e-14       1       False
70/15/15 quad_precision clap_general             muq        0.0338   0.0096  0.057  0.00205  0.0877       False
70/15/15 quad_precision clap_general           sbert        0.0417  0.00497 0.0805    0.102   0.336       False
70/15/15           ndcg clap_general clap_music_rand         0.178    0.151  0.205 1.17e-25       1       False
70/15/15           ndcg clap_general  clap_music_det         0.177    0.151  0.203 3.88e-27       1       False
70/15/15           ndcg clap_general     handcrafted        0.0982   0.0746  0.122 3.09e-14       1     

## 4. Late fusion with SBERT

In [5]:
fus_rows, fus_pq = [], {}
for lbl in SCHEMES.values():
    ct = selected[(lbl, 'sbert')]
    for a in ['clap_general', 'clap_music_det', 'muq']:
        ca = selected[(lbl, a)]
        al = max(ALPHAS, key=lambda x: run(('late', ca, ct, x), shared[lbl]['validate'])['quad_precision'].mean())
        m = run(('late', ca, ct, al), shared[lbl]['test']); fus_pq[(lbl, a)] = m
        row = {'split': lbl, 'audio': a, 'alpha': al}
        for metric in ['quad_precision', 'ndcg']:
            lo, hi = boot_ci(m[metric]); row.update({metric: m[metric].mean(), f'{metric}_lo': lo, f'{metric}_hi': hi})
        fus_rows.append(row)
fusion = pd.DataFrame(fus_rows)

# Tests: CLAP-general fusion vs lyrics alone, vs CLAP-general alone, vs MuQ + SBERT fusion (Holm per split × metric)
ft = []
for lbl in SCHEMES.values():
    for metric in ['quad_precision', 'ndcg']:
        x = fus_pq[(lbl, 'clap_general')][metric]
        block = []
        for name, y in [('lyrics alone', pq[(lbl, 'sbert')][metric]), ('clap_general alone', pq[(lbl, 'clap_general')][metric]),
                        ('MuQ + SBERT fusion', fus_pq[(lbl, 'muq')][metric])]:
            d = x - y; lo, hi = boot_ci(d)
            p = wilcoxon(x, y, zero_method='pratt').pvalue if (d != 0).any() else 1.0
            block.append({'split': lbl, 'metric': metric, 'clap_general+sbert vs': name, 'diff': d.mean(),
                          'ci_lo': lo, 'ci_hi': hi, 'p': p})
        for r, ph in zip(block, holm([r['p'] for r in block])):
            r['p_holm'] = ph
        ft += block
fusion_tests = pd.DataFrame(ft)
fusion.to_csv(os.path.join(OUT_DIR, 'clap_general_fusion.csv'), index=False)
fusion_tests.to_csv(os.path.join(OUT_DIR, 'clap_general_fusion_tests.csv'), index=False)
print(fusion.round(3).to_string(index=False)); print()
print(fusion_tests.to_string(index=False, float_format=lambda x: f'{x:.3g}'))

   split          audio  alpha  quad_precision  quad_precision_lo  quad_precision_hi  ndcg  ndcg_lo  ndcg_hi
70/15/15   clap_general    0.4           0.705              0.672              0.736 0.479    0.451    0.508
70/15/15 clap_music_det    0.3           0.609              0.575              0.644 0.425    0.396    0.453
70/15/15            muq    0.6           0.690              0.658              0.722 0.471    0.444    0.500
40/30/30   clap_general    0.5           0.724              0.702              0.745 0.495    0.475    0.515
40/30/30 clap_music_det    0.2           0.636              0.613              0.659 0.440    0.420    0.459
40/30/30            muq    0.6           0.696              0.674              0.717 0.478    0.459    0.497

   split         metric clap_general+sbert vs    diff    ci_lo  ci_hi        p   p_holm
70/15/15 quad_precision          lyrics alone   0.103   0.0768   0.13 1.09e-12 3.26e-12
70/15/15 quad_precision    clap_general alone  0.0613   0.03

## 5. Probes (70/15/15): ridge AV R² and logistic quadrant accuracy

In [6]:
msk = {s: np.isin(cat_sids, shared['70/15/15'][s]) for s in ['train', 'validate', 'test']}
tr, va, te = msk['train'], msk['validate'], msk['test']
RIDGE_ALPHAS, C_GRID = np.logspace(-2, 5, 15), np.logspace(-4, 1, 6)

def ridge_fit(X):
    p = make_pipeline(StandardScaler(), RidgeCV(alphas=RIDGE_ALPHAS)).fit(X[tr], Y[tr])
    return p, r2_score(Y[va], p.predict(X[va]))

def logreg_fit(X):
    best = None
    for Cc in C_GRID:
        clf = make_pipeline(StandardScaler(), LogisticRegression(C=Cc, max_iter=3000)).fit(X[tr], quad[tr])
        f1 = f1_score(quad[va], clf.predict(X[va]), average='macro')
        if best is None or f1 > best[0]:
            best = (f1, clf)
    return best

probe_rows = []
for fam in FAMS:
    reps = [r for r in REPS if FAMILY(r) == fam]
    rf = {r: ridge_fit(REPS[r]) for r in reps}; lf = {r: logreg_fit(REPS[r]) for r in reps}
    rr = max(reps, key=lambda r: rf[r][1]); rl = max(reps, key=lambda r: lf[r][0])
    P = rf[rr][0].predict(REPS[rr][te]); pred = lf[rl][1].predict(REPS[rl][te])
    corr = (pred == quad[te]).astype(float); lo, hi = boot_ci(corr)
    probe_rows.append({'family': fam, 'ridge_rep': rr, 'R2_arousal': r2_score(Y[te, 0], P[:, 0]),
                       'R2_valence': r2_score(Y[te, 1], P[:, 1]), 'logreg_rep': rl, 'quad_acc': corr.mean(),
                       'acc_lo': lo, 'acc_hi': hi, 'macro_f1': f1_score(quad[te], pred, average='macro')})
probes = pd.DataFrame(probe_rows)
probes.to_csv(os.path.join(OUT_DIR, 'clap_general_probes.csv'), index=False)
print(probes.round(3).to_string(index=False))

         family       ridge_rep  R2_arousal  R2_valence      logreg_rep  quad_acc  acc_lo  acc_hi  macro_f1
   clap_general    clap_general       0.513       0.417    clap_general     0.728   0.679   0.778     0.728
clap_music_rand clap_music_rand       0.199       0.080 clap_music_rand     0.450   0.394   0.510     0.458
 clap_music_det  clap_music_det       0.299       0.106  clap_music_det     0.500   0.444   0.556     0.506
    handcrafted     handcrafted       0.434       0.158     handcrafted     0.603   0.546   0.656     0.603
            muq          muq_L7       0.515       0.411          muq_L6     0.765   0.715   0.811     0.764
          sbert           sbert       0.373       0.519           sbert     0.722   0.669   0.772     0.718
